# Final Project

# Training Pipeline

In [1]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.3 MB/s eta 0:00:00


In [1]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path("/content/drive/MyDrive/DL_final_competition")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE        = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
GPU: Tesla T4


In [7]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

# print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
# train_df.head(2)

In [8]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def short_text(x, max_chars=1200):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    return x[:max_chars]

def build_prompt(row, include_answer=False):
    hint = short_text(row.get("hint", ""), max_chars=1000)
    lecture = short_text(row.get("lecture", ""), max_chars=600)

    subject = short_text(row.get("subject", ""), max_chars=100)
    topic = short_text(row.get("topic", ""), max_chars=150)
    skill = short_text(row.get("skill", ""), max_chars=150)

    choices = row["choices"]
    choices_str = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    prompt += "Choose the correct answer. Reply with only one letter.\n\n"

    # optional metadata
    if subject or topic:
        prompt += "Metadata:\n"
        if subject:
            prompt += f"Subject: {subject}\n"
        if topic:
            prompt += f"Topic: {topic}\n"
        prompt += "\n"

    # important context
    if skill:
        prompt += f"Skill / Strategy: {skill}\n\n"

    if hint:
        prompt += f"Passage / Hint:\n{hint}\n\n"

    if lecture:
        prompt += f"Background:\n{lecture}\n\n"

    prompt += f"Question:\n{row['question']}\n\n"
    prompt += f"Choices:\n{choices_str}\n\n"
    prompt += "Answer:"

    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"

    return prompt

In [9]:
class ScienceQADataset(Dataset):
    def __init__(self, df, is_train=True, image_map=None):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.image_map = image_map

    def __len__(self):
        return len(self.df)

    def _find_image_path(self, row):
        name1 = Path(row["image_path"]).name
        name2 = f"{row['id']}.png"

        for name in [name1, name2]:
            if self.image_map is not None and name in self.image_map:
                return self.image_map[name]

        actual_path = Path(f"data/images/{'train' if self.is_train else 'test'}") / name1
        if actual_path.exists():
            return actual_path

        raise FileNotFoundError(f"cannot find the plot of id={row['id']}")

    def _load_image(self, row):
        img_path = self._find_image_path(row)
        return Image.open(img_path).convert("RGB")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row)

        res = {
            "image": img,
            "id": row["id"],
            "choices": row["choices"]
        }

        if self.is_train:
            res["text"] = build_prompt(row, include_answer=True)
            res["answer_label"] = CHOICE_LETTERS[int(row["answer"])]
        else:
            res["text"] = build_prompt(row, include_answer=False)
            res["answer"] = int(row["answer"]) if "answer" in self.df.columns else -1

        return res

In [10]:
from pathlib import Path

TRAIN_IMG_ROOT = Path("/content/drive/MyDrive/DL_final_competition/images/images/train")
VAL_IMG_ROOT   = Path("/content/drive/MyDrive/DL_final_competition/images/images/val")
TEST_IMG_ROOT  = Path("/content/drive/MyDrive/DL_final_competition/images/images/test")

def build_image_map(root):
    imgs = []
    for ext in ["*.png", "*.jpg", "*.jpeg"]:
        imgs.extend(root.rglob(ext))
    image_map = {p.name: p for p in imgs}
    print(root, "num images:", len(image_map))
    print(list(image_map.items())[:3])
    return image_map

train_image_map = build_image_map(TRAIN_IMG_ROOT)
val_image_map   = build_image_map(VAL_IMG_ROOT)
test_image_map  = build_image_map(TEST_IMG_ROOT)

/content/drive/MyDrive/DL_final_competition/images/images/train num images: 3109
[('train_08377.png', PosixPath('/content/drive/MyDrive/DL_final_competition/images/images/train/train_08377.png')), ('train_08222.png', PosixPath('/content/drive/MyDrive/DL_final_competition/images/images/train/train_08222.png')), ('train_08584.png', PosixPath('/content/drive/MyDrive/DL_final_competition/images/images/train/train_08584.png'))]
/content/drive/MyDrive/DL_final_competition/images/images/val num images: 1048
[('val_00227.png', PosixPath('/content/drive/MyDrive/DL_final_competition/images/images/val/val_00227.png')), ('val_00069.png', PosixPath('/content/drive/MyDrive/DL_final_competition/images/images/val/val_00069.png')), ('val_00191.png', PosixPath('/content/drive/MyDrive/DL_final_competition/images/images/val/val_00191.png'))]
/content/drive/MyDrive/DL_final_competition/images/images/test num images: 1008
[('test_00197.png', PosixPath('/content/drive/MyDrive/DL_final_competition/images/imag

In [11]:
IMG_SIZE = 384

train_ds = ScienceQADataset(train_df, is_train=True, image_map=train_image_map)
val_ds   = ScienceQADataset(val_df, is_train=False, image_map=val_image_map)
test_ds  = ScienceQADataset(test_df, is_train=False, image_map=test_image_map)

print(len(train_ds), len(val_ds), len(test_ds))

3109 1048 1008


In [7]:
TRAIN_CFG = {
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj", "o_proj"],

    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "num_epochs": 1,

    "batch_size": 2,
    "gradient_accumulation_steps": 8,

    "max_train_samples": 2000,

    "warmup_ratio": 0.05,
    "max_grad_norm": 1.0,
    "lr_scheduler_type": "cosine",
}

In [9]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

if device == "cuda":
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=TRAIN_CFG["lora_r"],
    lora_alpha=TRAIN_CFG["lora_alpha"],
    lora_dropout=TRAIN_CFG["lora_dropout"],
    target_modules=TRAIN_CFG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")
assert trainable_params <= 5_000_000, f"Exceed the limitation: {trainable_params}"

print("Model & LoRA Ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Trainable parameters: 3,211,264
Model & LoRA Ready.


In [12]:
model.config.use_cache = False
model.gradient_checkpointing_enable()

In [13]:
def train_collate_fn(batch):
    images = [item["image"] for item in batch]
    texts = [item["text"] for item in batch]

    inputs = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )

    labels = inputs["input_ids"].clone()

    labels[labels == processor.tokenizer.pad_token_id] = -100

    target_ids = processor.tokenizer("Answer:", add_special_tokens=False)["input_ids"]

    for i in range(labels.shape[0]):
        curr_input_ids = inputs["input_ids"][i].tolist()

        match_pos = -1
        for j in range(len(curr_input_ids) - len(target_ids), -1, -1):
            if curr_input_ids[j : j + len(target_ids)] == target_ids:
                match_pos = j
                break

        if match_pos != -1:
            labels[i, : match_pos + len(target_ids)] = -100
        else:
            labels[i, :] = -100

    inputs["labels"] = labels
    return inputs

In [14]:
from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW

num_train = min(TRAIN_CFG["max_train_samples"], len(train_ds))
small_train_ds = Subset(train_ds, range(num_train))

train_loader = DataLoader(
    small_train_ds,
    batch_size=TRAIN_CFG["batch_size"],
    shuffle=True,
    collate_fn=train_collate_fn,
    num_workers=2,
    pin_memory=True
)

val_ds_for_loss = ScienceQADataset(
    val_df,
    is_train=True,
    image_map=val_image_map
)

val_loss_loader = DataLoader(
    val_ds_for_loss,
    batch_size=TRAIN_CFG["batch_size"],
    shuffle=False,
    collate_fn=train_collate_fn,
    num_workers=2,
    pin_memory=True
)

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=TRAIN_CFG["learning_rate"],
    weight_decay=TRAIN_CFG["weight_decay"],
)

#--------------------------------------------------------- add for scheduler--------------------------
# from transformers import get_scheduler
# import math

# steps_per_epoch = math.ceil(len(train_loader) / TRAIN_CFG["gradient_accumulation_steps"])
# total_steps = steps_per_epoch * TRAIN_CFG["num_epochs"]
# warmup_steps = int(total_steps * TRAIN_CFG["warmup_ratio"])

# scheduler = get_scheduler(
#     name=TRAIN_CFG["lr_scheduler_type"],
#     optimizer=optimizer,
#     num_warmup_steps=warmup_steps,
#     num_training_steps=total_steps,
# )

NameError: name 'TRAIN_CFG' is not defined

In [13]:
# calculate val loss
def evaluate_val_loss(model, val_loader, max_batches=30):
    model.eval()
    losses = []

    with torch.inference_mode():
        for step, batch in enumerate(val_loader):
            if step >= max_batches:
                break

            batch = {
                k: v.to(model.device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }

            outputs = model(**batch)
            loss = outputs.loss

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"Bad val loss at batch {step}: {loss}")
                continue

            losses.append(loss.item())

    model.train()

    if len(losses) == 0:
        return None

    return sum(losses) / len(losses)

In [ ]:
# train
from tqdm.auto import tqdm
train_loss_history = []
epoch_train_losses = []
epoch_val_losses = []

model.train()
optimizer.zero_grad()

for epoch in range(TRAIN_CFG["num_epochs"]):
    total_loss = 0.0
    step_count = 0

    for step, batch in enumerate(tqdm(train_loader)):
        batch = {
            k: v.to(model.device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        outputs = model(**batch)
        raw_loss = outputs.loss

        if torch.isnan(raw_loss) or torch.isinf(raw_loss):
            print(f"Bad training loss at step {step}: {raw_loss}")
            break

        loss = raw_loss / TRAIN_CFG["gradient_accumulation_steps"]
        loss.backward()

        if (step + 1) % TRAIN_CFG["gradient_accumulation_steps"] == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0 # 1.0
            )
            optimizer.step()
            # scheduler.step() # new_add
            optimizer.zero_grad()

        total_loss += raw_loss.item()
        train_loss_history.append(raw_loss.item())
        step_count += 1

        if step % 20 == 0:
            print(f"step {step} | train loss: {raw_loss.item():.4f}")

    avg_train_loss = total_loss / max(1, step_count)
    epoch_train_losses.append(avg_train_loss)

    val_loss = evaluate_val_loss(model, val_loss_loader, max_batches=30)
    epoch_val_losses.append(val_loss)

    print(f"Epoch {epoch+1}")
    print(f"Average train loss: {avg_train_loss:.4f}")
    print(f"Validation loss: {val_loss}")

  0%|          | 0/1000 [00:00<?, ?it/s]

step 0 | train loss: 0.2594
step 20 | train loss: 0.4691
step 40 | train loss: 1.3946
step 60 | train loss: 0.6274
step 80 | train loss: 0.0172
step 100 | train loss: 0.5114
step 120 | train loss: 1.0717
step 140 | train loss: 0.1690
step 160 | train loss: 0.1221
step 180 | train loss: 0.6737
step 200 | train loss: 0.6660
step 220 | train loss: 0.0487
step 240 | train loss: 0.0623
step 260 | train loss: 0.1193
step 280 | train loss: 0.0343
step 300 | train loss: 0.0015
step 320 | train loss: 0.0962
step 340 | train loss: 0.1868
step 360 | train loss: 0.9227
step 380 | train loss: 0.6591
step 400 | train loss: 0.4105
step 420 | train loss: 0.3036
step 440 | train loss: 0.4940
step 460 | train loss: 0.0622
step 480 | train loss: 0.3472
step 500 | train loss: 0.4663
step 520 | train loss: 1.8721
step 540 | train loss: 0.0390
step 560 | train loss: 0.5366
step 580 | train loss: 0.2494
step 600 | train loss: 0.4698
step 620 | train loss: 0.5532
step 640 | train loss: 0.3203
step 660 | train

In [ ]:
import os

save_directory = "/content/drive/MyDrive/DL_final_competition/lora_weights_0501_1"

os.makedirs(save_directory, exist_ok=True)

model.save_pretrained(save_directory)

processor.save_pretrained(save_directory)

print(f"Success to save: {save_directory}")

✅ LoRA 權重已成功儲存至: /content/drive/MyDrive/DL_final_competition/lora_weights_0501_1


# Validation Pipeline

In [15]:
import re

def extract_answer(text: str, num_choices: int) -> str:
    valid_letters = CHOICE_LETTERS[:num_choices]

    if "Answer:" in text:
        text = text.split("Answer:")[-1]

    match = re.search(r"\b([A-J])\b", text)
    if match:
        pred = match.group(1)
        if pred in valid_letters:
            return pred

    for ch in valid_letters:
        if ch in text:
            return ch

    return "A"

In [16]:
def predict_one(model, processor, item):
    model.eval()

    inputs = processor(
        text=[item["text"]],
        images=[item["image"]],
        return_tensors="pt",
    )

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
        )

    decoded = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    pred_letter = extract_answer(decoded, num_choices=len(item["choices"]))
    return pred_letter, decoded

In [17]:
from tqdm import tqdm
from collections import Counter

def evaluate_accuracy(model, processor, val_ds, max_samples=100):
    correct = 0
    total = 0
    preds = []

    n = min(max_samples, len(val_ds))

    for i in tqdm(range(n)):
        item = val_ds[i]
        pred_letter, raw_output = predict_one(model, processor, item)

        pred_idx = CHOICE_LETTERS.index(pred_letter)
        true_idx = item["answer"]

        correct += int(pred_idx == true_idx)
        total += 1
        preds.append(pred_letter)

        if i < 5:
            print("=" * 80)
            print("Prediction:", pred_letter)
            print("True:", CHOICE_LETTERS[true_idx])
            print("Raw output:", raw_output[-300:])

    acc = correct / total
    print(f"Validation accuracy on {total} samples: {acc:.4f}")
    print("Prediction distribution:", Counter(preds))

    return acc

In [ ]:
val_acc = evaluate_accuracy(model, processor, val_ds, max_samples=300)

  0%|          | 1/300 [00:02<10:58,  2.20s/it]

Prediction: B
True: A
Raw output: snail leech? Complete the claim below that answers this question and is best supported by the passage.
Covering its eggs with its body increases the chances that ().

Choices:
A. the leech's eggs will hatch
B. the leech will not eat for up to a week
C. the leech will fight a water snail

Answer: B




  1%|          | 2/300 [00:04<09:54,  2.00s/it]

Prediction: A
True: B
Raw output: claim below that answers this question and is best supported by the passage.
Fanning eggs increases the chances that ().

Choices:
A. the male will build a nest for females to lay eggs in
B. the male's offspring will become adults
C. the male will spend more energy while waving his fins

Answer: A




  1%|          | 3/300 [00:05<09:34,  1.93s/it]

...

Prediction: B
True: A
Raw output: ozygous for that gene.
In a Punnett square, each box represents a different outcome, or result. Each of the

Question:
What is the probability that a Cepaea snail produced by this cross will be homozygous recessive for the shell banding gene?

Choices:
A. 2/4
B. 4/4
C. 0/4
D. 3/4
E. 1/4

Answer: B




  2%|▏         | 5/300 [00:10<10:57,  2.23s/it]

Prediction: B
True: E
Raw output: zygous for that gene.
In a Punnett square, each box represents a different outcome, or result. Each of the

Question:
What is the probability that a Cepaea snail produced by this cross will be homozygous recessive for the shell banding gene?

Choices:
A. 2/4
B. 0/4
C. 3/4
D. 1/4
E. 4/4

Answer: B





100%|██████████| 300/300 [11:08<00:00,  2.23s/it]

Validation accuracy on 300 samples: 0.6867
Prediction distribution: Counter({'B': 118, 'A': 108, 'C': 57, 'D': 17})


In [18]:
lora_params = [name for name, p in model.named_parameters() if "lora" in name.lower()]
print(len(lora_params))
print(lora_params[:10])

240
['base_model.model.model.vision_model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.vision_model.encoder.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.vision_model.encoder.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.vision_model.encoder.layers.1.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_model.encoder.layers.1.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.vision_model.encoder.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.vision_model.encoder.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.vision_model.encoder.layers.2.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_model.encoder.layers.2.self_attn.v_proj.lora_B.default.weight']


# Inference Pipeline

### During experimentation, inference was first performed directly using the fine-tuned model after training, while this LoRA reloading inference pipeline is provided only for reproducibility so that predictions can be regenerated without retraining.

In [2]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel
import torch

BASE_MODEL_ID = MODEL_ID
LORA_DIR = "/content/drive/MyDrive/DL_final_competition/lora_weights_0501_1"

processor = AutoProcessor.from_pretrained(LORA_DIR)

base_model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

print("LoRA loaded")
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


LoRA loaded
trainable params: 0 || all params: 510,693,568 || trainable%: 0.0000


In [3]:
import re

def extract_answer(text: str, num_choices: int) -> str:
    valid_letters = CHOICE_LETTERS[:num_choices]

    if "Answer:" in text:
        text = text.split("Answer:")[-1]

    match = re.search(r"\b([A-J])\b", text)
    if match:
        pred = match.group(1)
        if pred in valid_letters:
            return pred

    for ch in valid_letters:
        if ch in text:
            return ch

    return "A"

In [4]:
def predict_one(model, processor, item):
    model.eval()

    inputs = processor(
        text=[item["text"]],
        images=[item["image"]],
        return_tensors="pt",
    )

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
        )

    decoded = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    pred_letter = extract_answer(decoded, num_choices=len(item["choices"]))
    return pred_letter, decoded

In [5]:
from tqdm import tqdm
from collections import Counter

def evaluate_accuracy(model, processor, val_ds, max_samples=100):
    correct = 0
    total = 0
    preds = []

    n = min(max_samples, len(val_ds))

    for i in tqdm(range(n)):
        item = val_ds[i]
        pred_letter, raw_output = predict_one(model, processor, item)

        pred_idx = CHOICE_LETTERS.index(pred_letter)
        true_idx = item["answer"]

        correct += int(pred_idx == true_idx)
        total += 1
        preds.append(pred_letter)

        if i < 5:
            print("=" * 80)
            print("Prediction:", pred_letter)
            print("True:", CHOICE_LETTERS[true_idx])
            print("Raw output:", raw_output[-300:])

    acc = correct / total
    print(f"Validation accuracy on {total} samples: {acc:.4f}")
    print("Prediction distribution:", Counter(preds))

    return acc

In [15]:
from tqdm import tqdm
from collections import Counter
import pandas as pd

LETTER_TO_IDX = {ch: i for i, ch in enumerate(CHOICE_LETTERS)}

def generate_submission(model, processor, test_ds, test_df, output_path="submission.csv", max_samples=None):
    rows = []
    preds = []

    n = len(test_ds) if max_samples is None else min(max_samples, len(test_ds))

    model.eval()

    for i in tqdm(range(n)):
        item = test_ds[i]

        # same as validation pipeline
        pred_letter, raw_output = predict_one(model, processor, item)

        pred_idx = LETTER_TO_IDX.get(pred_letter, 0)
        preds.append(pred_letter)

        rows.append({
            "id": test_df.iloc[i]["id"],
            "answer": pred_idx,
        })

        if i < 5:
            print("=" * 80)
            print("ID:", test_df.iloc[i]["id"])
            print("Prediction:", pred_letter)
            print("Pred index:", pred_idx)
            print("Raw output:", raw_output[-300:])

    submission = pd.DataFrame(rows)
    submission.to_csv(output_path, index=False)

    print(f"Saved {output_path}")
    print("Num predictions:", len(submission))
    print("Prediction distribution:", Counter(preds))
    print(submission.head())

    return submission

In [ ]:
submission_test = generate_submission(
    model,
    processor,
    test_ds,
    test_df,
    output_path="submission2.csv",
    max_samples=None
)

  0%|          | 1/1008 [00:01<30:57,  1.84s/it]

ID: test_01750
Prediction: B
Pred index: 1
Raw output: d farmers have appreciated cats eight thousand years ago?

Choices:
A. The cats were thought to be visiting goddesses.
B. The cats hunted and brought food to the farmers.
C. The cats helped keep the farmers' grain free of mice.
D. The cats helped farmers find better places to store grain.

Answer: B


  0%|          | 2/1008 [00:04<35:19,  2.11s/it]

ID: test_00128
Prediction: A
Pred index: 0
Raw output: erozygous for that gene.
In a Punnett square, each box represents a different outcome, or result. Each of the

Question:
What is the probability that an American curl cat produced by this cross will be homozygous dominant for the ear type gene?

Choices:
A. 0/4
B. 2/4
C. 4/4
D. 3/4
E. 1/4

Answer: A


  0%|          | 3/1008 [00:06<36:53,  2.20s/it]

...

ID: test_02425
Prediction: B
Pred index: 1
Raw output: us for that gene.
In a Punnett square, each box represents a different outcome, or result. Each of the

Question:
What is the probability that a budgerigar parakeet produced by this cross will be heterozygous for the body feather color gene?

Choices:
A. 4/4
B. 2/4
C. 3/4
D. 1/4
E. 0/4

Answer: B





  0%|          | 5/1008 [00:11<39:58,  2.39s/it]

ID: test_00930
Prediction: C
Pred index: 2
Raw output: ygous for that gene.
In a Punnett square, each box represents a different outcome, or result. Each of the

Question:
What is the probability that a muskmelon plant produced by this cross will be homozygous recessive for the fruit taste gene?

Choices:
A. 2/4
B. 1/4
C. 0/4
D. 3/4
E. 4/4

Answer: C





100%|██████████| 1008/1008 [39:26<00:00,  2.35s/it]

Saved submission2.csv
Num predictions: 1008
Prediction distribution: Counter({'A': 391, 'B': 325, 'C': 238, 'D': 54})
           id  answer
0  test_01750       1
1  test_00128       0
2  test_02891       1
3  test_02425       1
4  test_00930       2


In [16]:
print(len(test_ds), len(test_df))

1008 1008
